# End-to-End Machine Learning Pipeline for Predicting Spotify Song Popularity

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [ ]:
# Load Spotify dataset
df = pd.read_csv("spotify_tracks.csv")

# Inspect dataset structure
df.head()

# Basic dataset information
df.info()
df.describe()

In [ ]:
# Select relevant audio features
features = [
    "danceability",
    "energy",
    "loudness",
    "tempo",
    "valence",
    "acousticness",
    "speechiness"
]

X = df[features]

# Convert popularity into binary classification target
# Example: songs above median popularity = popular
threshold = df["popularity"].median()

y = (df["popularity"] > threshold).astype(int)

In [ ]:
# Correlation matrix
plt.figure(figsize=(10,6))
sns.heatmap(df[features + ["popularity"]].corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()

# Distribution of key audio features
df[features].hist(figsize=(12,8))
plt.suptitle("Distribution of Audio Features")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=77,
    stratify=y
)

In [ ]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_pipeline.fit(X_train, y_train)

y_pred = logistic_pipeline.predict(X_test)
y_prob = logistic_pipeline.predict_proba(X_test)[:,1]

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_split=20,
    random_state=77
)

tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, tree_pred))

In [ ]:
cv_scores = cross_val_score(
    logistic_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross-Validation Accuracy:", np.mean(cv_scores))

In [ ]:
importance = tree_model.feature_importances_

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": importance
}).sort_values(by="importance", ascending=False)

print(feature_importance)

In [ ]:
sns.barplot(
    data=feature_importance,
    x="importance",
    y="feature"
)

plt.title("Feature Importance (Decision Tree)")
plt.show()

In [ ]:
print(classification_report(y_test, y_pred))